In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("debug-bronze-read")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
orders_schema = StructType([
    StructField("event_type", StringType(), True),
    StructField("invoice_id", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("event_time", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("ingestion_time", StringType(), True)
])

In [2]:
df = spark.read.parquet("s3a://bronze/orders/")
df.show(5)

+---------+--------------------+-------------------+---------+------+--------------------+---------------------+
|kafka_key|            raw_json|              topic|partition|offset|     kafka_timestamp|bronze_ingestion_time|
+---------+--------------------+-------------------+---------+------+--------------------+---------------------+
|   489436|{"event_type": "o...|retail_order_events|        0|     0|2026-04-07 12:23:...| 2026-04-07 12:26:...|
|   489436|{"event_type": "o...|retail_order_events|        0|     1|2026-04-07 12:23:...| 2026-04-07 12:26:...|
|   489436|{"event_type": "o...|retail_order_events|        0|     2|2026-04-07 12:23:...| 2026-04-07 12:26:...|
|   489436|{"event_type": "o...|retail_order_events|        0|     3|2026-04-07 12:23:...| 2026-04-07 12:26:...|
|   489436|{"event_type": "o...|retail_order_events|        0|     4|2026-04-07 12:23:...| 2026-04-07 12:26:...|
+---------+--------------------+-------------------+---------+------+--------------------+------

In [7]:
from pyspark.sql.functions import from_json, col

df_parsed = df.withColumn("parsed", from_json(col("raw_json"), orders_schema))
df_parsed.select("parsed.*").show(5)

+------------------+----------+----------+--------------------+--------+----------+--------------+-----------+-------------------+------------------+--------------------+
|        event_type|invoice_id|stock_code|         description|quantity|unit_price|       country|customer_id|         event_time|      total_amount|      ingestion_time|
+------------------+----------+----------+--------------------+--------+----------+--------------+-----------+-------------------+------------------+--------------------+
|order_item_created|    489436|     22194|BLACK DINER WALL ...|       2|       8.5|United Kingdom|      13078|2009-12-01 09:06:00|              17.0|2026-04-07 12:23:...|
|order_item_created|    489436|    84596L|BISCUITS SMALL BO...|       8|      1.25|United Kingdom|      13078|2009-12-01 09:06:00|              10.0|2026-04-07 12:23:...|
|order_item_created|    489436|    84596F|SMALL MARSHMALLOW...|       8|      1.25|United Kingdom|      13078|2009-12-01 09:06:00|              1

In [12]:
df_parsed.select("parsed.*").show(5)

+------------------+----------+----------+--------------------+--------+----------+--------------+-----------+-------------------+------------------+--------------------+
|        event_type|invoice_id|stock_code|         description|quantity|unit_price|       country|customer_id|         event_time|      total_amount|      ingestion_time|
+------------------+----------+----------+--------------------+--------+----------+--------------+-----------+-------------------+------------------+--------------------+
|order_item_created|    489436|     22194|BLACK DINER WALL ...|       2|       8.5|United Kingdom|      13078|2009-12-01 09:06:00|              17.0|2026-04-07 12:23:...|
|order_item_created|    489436|    84596L|BISCUITS SMALL BO...|       8|      1.25|United Kingdom|      13078|2009-12-01 09:06:00|              10.0|2026-04-07 12:23:...|
|order_item_created|    489436|    84596F|SMALL MARSHMALLOW...|       8|      1.25|United Kingdom|      13078|2009-12-01 09:06:00|              1

In [32]:
df_parsed.rdd.getNumPartitions()

940

In [13]:
df_parsed.columns

['kafka_key',
 'raw_json',
 'topic',
 'partition',
 'offset',
 'kafka_timestamp',
 'bronze_ingestion_time',
 'parsed']

In [15]:
parsed_df_values = df_parsed.select("parsed.*")

In [18]:
df_parsed.filter(col("parsed").isNull()).count()

0

In [21]:
row_count = df_parsed.count()

In [23]:
row_count

525425

In [24]:
df_test = df_parsed.repartition(20)
df_test.rdd.getNumPartitions()

20

In [25]:
df_test.write.mode("overwrite").parquet("s3a://silver/test/")

In [26]:
df_new = spark.read.parquet("s3a://silver/test/")

In [27]:
df_new.rdd.getNumPartitions()

10

In [31]:
len(df_new.inputFiles())

20